In [3]:
import pandas as pd
import numpy as nd

# Collection and cleaning

In [4]:
#read one of the files, and set the ride_id as index/primary key.
r= pd.read_csv('data1.csv')

In [5]:
#list of file names
datas = ['data1.csv', 'data2.csv', 'data3.csv', 'data4.csv', 'data5.csv', 'data6.csv', 'data7.csv', 'data8.csv', 'data9.csv', 'data10.csv', 'data11.csv', 'data12.csv']
dfs = []
#make a list of all the files(read already)
for data in datas:
    df = pd.read_csv(data)
    dfs.append(df)

rd = pd.concat(dfs, ignore_index=True)

In [6]:
rd = rd.set_index('ride_id')

In [7]:
#we use .info to get concise information about the whole dataset
#using info will help me understand the datatypes, along with understanding anamolies, like null values and such.
rd.info(show_counts = True)

<class 'pandas.core.frame.DataFrame'>
Index: 5552994 entries, 7569BC890583FCD7 to D2CF23B6D79232B6
Data columns (total 12 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   rideable_type       5552994 non-null  object 
 1   started_at          5552994 non-null  object 
 2   ended_at            5552994 non-null  object 
 3   start_station_name  4368321 non-null  object 
 4   start_station_id    4368321 non-null  object 
 5   end_station_name    4309689 non-null  object 
 6   end_station_id      4309689 non-null  object 
 7   start_lat           5552994 non-null  float64
 8   start_lng           5552994 non-null  float64
 9   end_lat             5547459 non-null  float64
 10  end_lng             5547459 non-null  float64
 11  member_casual       5552994 non-null  object 
dtypes: float64(4), object(8)
memory usage: 550.8+ MB


In [8]:
# obervations: dtypes are not appropriate, need datetime for started_at and ended_at.
# end gps coordinates are less that the start ones
# there are null values for start and end station names, and ids(but they are consistent amongst themselves(names = ids)
#checking if there is any relation between the missing start_station info and bike type.
rd[rd['start_station_name'].isnull()]['rideable_type'].value_counts()
# this reflects, electric bikes can be left anywhere basically, so we have to consider that usage patern too.

rideable_type
electric_bike    1184673
Name: count, dtype: int64

In [9]:
#now for the end stations:
rd[rd['end_station_name'].isnull()]['rideable_type'].value_counts()
#here there are both kinds of bikes, this could mean stolen bikes, or bikes on rides when the data is recorded

rideable_type
electric_bike    1237672
classic_bike        5633
Name: count, dtype: int64

In [10]:
#now we will deal with these anamolies,
#end coordinate null values only hold 0.099% data, hence we will delete these rows
##rd[rd['end_lat'].notnull()] - this is simple method, but below is the function to do so...
rd = rd.dropna(subset= ['end_lat'])
rd.info(show_counts = True)

<class 'pandas.core.frame.DataFrame'>
Index: 5547459 entries, 7569BC890583FCD7 to D2CF23B6D79232B6
Data columns (total 12 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   rideable_type       5547459 non-null  object 
 1   started_at          5547459 non-null  object 
 2   ended_at            5547459 non-null  object 
 3   start_station_name  4362786 non-null  object 
 4   start_station_id    4362786 non-null  object 
 5   end_station_name    4309689 non-null  object 
 6   end_station_id      4309689 non-null  object 
 7   start_lat           5547459 non-null  float64
 8   start_lng           5547459 non-null  float64
 9   end_lat             5547459 non-null  float64
 10  end_lng             5547459 non-null  float64
 11  member_casual       5547459 non-null  object 
dtypes: float64(4), object(8)
memory usage: 550.2+ MB


In [11]:
# changing the datatype of start and end_at to datetime for better working.
rd['started_at'] = pd.to_datetime(rd['started_at'])
rd['ended_at'] = pd.to_datetime(rd['ended_at'])

In [12]:
rd.info(show_counts = True)

<class 'pandas.core.frame.DataFrame'>
Index: 5547459 entries, 7569BC890583FCD7 to D2CF23B6D79232B6
Data columns (total 12 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   rideable_type       5547459 non-null  object        
 1   started_at          5547459 non-null  datetime64[ns]
 2   ended_at            5547459 non-null  datetime64[ns]
 3   start_station_name  4362786 non-null  object        
 4   start_station_id    4362786 non-null  object        
 5   end_station_name    4309689 non-null  object        
 6   end_station_id      4309689 non-null  object        
 7   start_lat           5547459 non-null  float64       
 8   start_lng           5547459 non-null  float64       
 9   end_lat             5547459 non-null  float64       
 10  end_lng             5547459 non-null  float64       
 11  member_casual       5547459 non-null  object        
dtypes: datetime64[ns](2), float64(4), object(6)
memory 

In [13]:
#we will find the duartion of each ride, to understand the trips better
rd['ride_duration'] = rd['ended_at']-rd['started_at']

In [14]:
# now the anamolies are handles, we will find the day of the week of the ride.
rd[['started_at', 'ended_at']]
rd['day_of_week'] = rd['started_at'].dt.day_name()

In [15]:
rd

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration,day_of_week
ride_id,,,,,,,,,,,,,,
7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member,0 days 00:13:57.477000,Tuesday
013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member,0 days 00:05:04.344000,Saturday
EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,0 days 00:11:35.500000,Thursday
EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:03:34.233000,Thursday
7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:02:34.429000,Thursday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53202484E8371237,electric_bike,2025-12-19 13:43:41.793,2025-12-19 13:54:07.779,Sheffield Ave & Kingsbury St,CHI02023,Halsted St & Melrose St,CHI02162,41.910522,-87.653106,41.941120,-87.649070,member,0 days 00:10:25.986000,Friday
3582BF76707AB2ED,classic_bike,2025-12-27 12:43:48.446,2025-12-27 12:52:45.141,Larrabee St & Webster Ave,CHI00454,Halsted St & Melrose St,CHI02162,41.921822,-87.644140,41.941120,-87.649070,member,0 days 00:08:56.695000,Saturday
B12D92DF229EFA41,electric_bike,2025-12-20 01:12:39.053,2025-12-20 01:18:55.928,Clarendon Ave & Junior Ter,CHI00395,Halsted St & Melrose St,CHI02162,41.961004,-87.649603,41.941120,-87.649070,casual,0 days 00:06:16.875000,Saturday


In [16]:
# converting ride duartion into minutes for my understanding
rd['ride_minutes'] = rd['ride_duration'].dt.total_seconds() / 60
rd

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration,day_of_week,ride_minutes
ride_id,,,,,,,,,,,,,,,
7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member,0 days 00:13:57.477000,Tuesday,13.957950
013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member,0 days 00:05:04.344000,Saturday,5.072400
EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,0 days 00:11:35.500000,Thursday,11.591667
EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:03:34.233000,Thursday,3.570550
7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:02:34.429000,Thursday,2.573817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53202484E8371237,electric_bike,2025-12-19 13:43:41.793,2025-12-19 13:54:07.779,Sheffield Ave & Kingsbury St,CHI02023,Halsted St & Melrose St,CHI02162,41.910522,-87.653106,41.941120,-87.649070,member,0 days 00:10:25.986000,Friday,10.433100
3582BF76707AB2ED,classic_bike,2025-12-27 12:43:48.446,2025-12-27 12:52:45.141,Larrabee St & Webster Ave,CHI00454,Halsted St & Melrose St,CHI02162,41.921822,-87.644140,41.941120,-87.649070,member,0 days 00:08:56.695000,Saturday,8.944917
B12D92DF229EFA41,electric_bike,2025-12-20 01:12:39.053,2025-12-20 01:18:55.928,Clarendon Ave & Junior Ter,CHI00395,Halsted St & Melrose St,CHI02162,41.961004,-87.649603,41.941120,-87.649070,casual,0 days 00:06:16.875000,Saturday,6.281250


In [17]:
#check max, min, mean, median values of ride duartions
rd[['ride_duration', 'ride_minutes']].agg(['max', 'min', 'mean', 'median'])

,ride_duration,ride_minutes
max,1 days 00:59:58.064000,1499.967733
min,-1 days +23:05:12.312000,-54.794800
mean,0 days 00:14:34.411429139,14.573524
median,0 days 00:09:24.998000,9.416633


In [18]:
# mean and median seem okay, lets see how many values are in negative
neg_count = (rd['ride_minutes'] < 0).sum()
zero_count = (rd['ride_minutes'] == 0).sum()
print(neg_count)
print(zero_count)

29
0


In [19]:
# here are only 29 rides with negative value, and no ride with zero value, i'll just delete these anomalies
rd = rd[rd['ride_minutes'] >= 1]
rd['ride_minutes'].min()

1.0

In [20]:
rd.head(5)

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration,day_of_week,ride_minutes
ride_id,,,,,,,,,,,,,,,
7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member,0 days 00:13:57.477000,Tuesday,13.957950
013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member,0 days 00:05:04.344000,Saturday,5.072400
EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,0 days 00:11:35.500000,Thursday,11.591667
EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:03:34.233000,Thursday,3.570550
7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:02:34.429000,Thursday,2.573817


In [21]:
rd.nlargest(10, 'ride_minutes')

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration,day_of_week,ride_minutes
ride_id,,,,,,,,,,,,,,,
1C849DDC34D1176E,classic_bike,2025-08-15 10:04:30.510,2025-08-16 11:04:28.574,Loomis St & Lexington St,CHI00238,NaN,NaN,41.872229,-87.661364,41.87,-87.64,casual,1 days 00:59:58.064000,Friday,1499.967733
EAD21D0D46746B60,classic_bike,2025-07-25 23:11:57.054,2025-07-27 00:11:55.013,Clark St & Newport St,CHI00674,NaN,NaN,41.944540,-87.654678,41.95,-87.65,member,1 days 00:59:57.959000,Friday,1499.965983
B092418B64D82DF0,classic_bike,2025-10-05 02:31:08.334,2025-10-06 03:31:05.649,Damen Ave & Pierce Ave,CHI02057,NaN,NaN,41.909358,-87.677758,41.96,-87.73,casual,1 days 00:59:57.315000,Sunday,1499.955250
C133E4D6A989E9D0,classic_bike,2025-11-16 18:29:34.644,2025-11-17 19:29:31.784,Halsted St & Clybourn Ave,CHI01161,NaN,NaN,41.909668,-87.648128,41.92,-87.65,casual,1 days 00:59:57.140000,Sunday,1499.952333
3E8C8CB63BEB4CC3,classic_bike,2025-11-16 13:44:35.456,2025-11-17 14:44:32.500,Sheffield Ave & Kingsbury St,CHI02023,NaN,NaN,41.910522,-87.653106,41.91,-87.68,casual,1 days 00:59:57.044000,Sunday,1499.950733
B9AAB0590C67FB33,classic_bike,2025-04-30 17:10:16.671,2025-05-01 18:10:13.341,Field Museum,13029,NaN,NaN,41.865312,-87.617867,41.88,-87.62,casual,1 days 00:59:56.670000,Wednesday,1499.944500
E21B2EC5E3C24218,classic_bike,2025-08-15 15:49:21.296,2025-08-16 16:49:17.699,Damen Ave & 87th St,CHI01761,NaN,NaN,41.736044,-87.672861,41.74,-87.67,casual,1 days 00:59:56.403000,Friday,1499.940050
E5EB8B98E1E489C6,classic_bike,2025-08-05 00:48:26.614,2025-08-06 01:48:22.879,Loomis St & 89th St,CHI00738,NaN,NaN,41.732380,-87.658069,41.73,-87.65,casual,1 days 00:59:56.265000,Tuesday,1499.937750
D389D7C1F3FA7AFF,classic_bike,2025-08-28 16:33:03.738,2025-08-29 17:32:59.744,Clark St & Bryn Mawr Ave,CHI00608,NaN,NaN,41.983593,-87.669154,41.98,-87.68,casual,1 days 00:59:56.006000,Thursday,1499.933433


In [22]:
#rides under 1 minute are really useless, this could mean mistaken docking/undocking.
#rides over 1 day will also be unconventional, so we remove them too.
rd = rd[(rd['ride_minutes'] >= 1) & (rd['ride_minutes'] <= 1440)]

# Analysis


In [23]:
#looking at for how long member/casual riders, ride with classic and electric bikes. 
rd.pivot_table(
    values='ride_minutes',
    index=['rideable_type', 'member_casual'],
    aggfunc=['mean', 'median', 'count']
)
# we can observe that the casual riders usually take longer rides than the members, also electic bikes coorporate for shorter rides than the classic bikes.

# We can observe that there is a bigger polarization between casual riders taking classic bike or electric(15 mean point difference), for longer trips
# than the polarization for the members riders (2 mean point difference)
# this could mean that for casual riders, classic bikes could be more economical,for longer rides, and for the members, the difference be minimal between the bikes
# usually casual riders take longer rides than the members - this could mean that rides are overall more economical for members, since they
# prefer the bikes even for shorter trips, while casual riders prefer to walk (or other commute) for shorter trips.


mean       median        count
                            ride_minutes ride_minutes ride_minutes
rideable_type member_casual                                       
classic_bike  casual           29.149262    16.278917       667873
              member           13.613222     9.212900      1274412
electric_bike casual           14.899408    10.240933      1247813
              member           11.346528     8.498717      2209751

In [24]:
# analyzing the distribution of casual and memeber riders, and the classic and electric bikes.
rd.groupby(['member_casual', 'rideable_type']).size() 
# there are double the member riders(~65%), to the casual ones, and double the electric bike rides(65%) overall,  
# moreover both member and casual riders reflect this share(~64% electric bike trips for each type)
# this stat even strengthens the point that even though there are double the electric bikes,
# electric bike rides are fairly shorter than classic bike rides(29 vs 14 mean points) for casual riders.
# this difference is far smaller for the members, even thought the share of electric bike rides is similar to casual members.
# so we can say for longer trips, casual members prefer classic bikes, and electric ones for shorter rides only.
# for the members, though they would prefer classic bikes for longer rides too, but the difference is minimal.

member_casual  rideable_type
casual         classic_bike      667873
               electric_bike    1247813
member         classic_bike     1274412
               electric_bike    2209751
dtype: int64

In [31]:
#now we move time analysis, we will see how the mean and median ride lengths work with day of the week
rd.pivot_table(
    values='ride_minutes',
    index=['member_casual', 'day_of_week'],
    aggfunc=['count', 'mean', 'median']
)
# this shows that rides are fairly longer on the weekend(fri, sat, sun) with casual riders usually taking longer rides, than their member counterparts.
# this could mean people taking trips, and outdoors across the weekend, in contrast to work commute on the weekends.
# Throughout the week the ride count for members is usually higher than the casuals but on weekdays casuals and members have a fairly equal rides. 

count         mean       median
                          ride_minutes ride_minutes ride_minutes
member_casual day_of_week                                       
casual        Friday            306453    19.589890    11.815833
              Monday            219224    19.680361    11.422533
              Saturday          395551    22.305530    13.768217
              Sunday            316929    23.063237    13.857917
              Thursday          247875    17.421031    10.584650
              Tuesday           216922    17.599607    10.518242
              Wednesday         212732    16.328191    10.182375
member        Friday            518541    12.136655     8.645617
              Monday            493317    11.755429     8.395933
              Saturday          439868    13.291370     9.425458
              Sunday            374532    13.441939     9.252067
              Thursday          565172    11.724510     8.615917
              Tuesday           552527    11.843950     8.628817
              Wednesday         540206    11.621486     8.555350

In [26]:
# analysis on basis of month 
rd['month'] = rd['started_at'].dt.month_name()
order = ['January', 'February', 'March', 'April', 'May', 'June',
         'July', 'August', 'September', 'October', 'November', 'December']
rd['month'] = pd.Categorical(rd['month'], categories=order, ordered=True)
rd.pivot_table(
    index='month',
    columns='member_casual',
    values='ride_minutes',
    aggfunc=['count',  'mean', 'median'],
    observed=False
)

# both casual and member riders ride more during the summer months, where ride count peaks around the months of june-july-august.
# Contrary to this, ride durations are fairly similar for members throughout the year.
# while the duration peaks at around the month of June, July, and August for the casuals.
# This may further strengthen the standpoint that for members longer or shorter rides are equal financially,
# while for casual riders longer rides are better economically than the shorter rides.
# There is a far bigger drop across august and december for casuals(x12) than the members(x4), 
# which could mean if a person pays for a membership, they are more resilient to seasons.

count               mean                median          
member_casual  casual  member     casual     member     casual    member
month                                                                   
January         23405  112331  11.973313   9.979503   7.137617  6.879067
February        27003  122097  12.381231   9.957846   7.382300  7.004050
March           82860  208457  17.848788  11.127660  10.568642  7.887283
April          105256  257919  18.630088  11.268450  10.707042  8.083383
May            175648  313994  20.799818  11.896704  12.264333  8.716975
June           278675  379517  21.860043  12.839433  13.378400  9.582917
July           308429  430392  21.426095  13.154796  12.983367  9.560233
August         323523  443125  21.695495  13.075886  13.446700  9.656800
September      254714  440951  19.544962  12.615738  11.957717  9.287967
October        214373  414088  18.080617  12.119433  10.713317  8.566442
November        94689  251912  15.079331  11.655116   8.950717  7.914775
December        27111  109380  12.921023  11.677574   7.678733  7.487492

In [27]:
# analysing on basis of time of the day
rd['hour'] = rd['started_at'].dt.hour
def time_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'
rd['time_of_day'] = rd['hour'].apply(time_of_day)
order = ['Morning', 'Afternoon', 'Evening', 'Night']
rd['time_of_day'] = pd.Categorical(rd['time_of_day'], categories=order, ordered=True)
rd.pivot_table(
    values='ride_minutes',
    index='time_of_day',
    columns='member_casual',
    aggfunc=['count', 'mean', 'median'],
    observed=False
)

# overall lesser rides are taken during the night time.
# for members, ride druations are distributed across the day.
# on the other hand, casuals take longer rides during the afternoon- could imply lunch breaks or work related rides.
# this furthur strengthens the point, that the rides are economical only for longer durations for the casuals, while the difference is minimal for members

count                mean                median          
member_casual  casual   member     casual     member     casual    member
time_of_day                                                              
Morning        408650  1051389  19.888348  11.376230  10.516850  8.162417
Afternoon      700481  1115254  21.961820  12.470857  13.517300  8.840800
Evening        540366  1003958  18.609134  12.718602  11.949742  9.323208
Night          266189   313562  16.877953  12.067459  10.157217  8.591300

# key insights

## Observations:
1. Casual riders take longer-duration rides than the members.
2. Even though the number of electric bike rides is double the number of classic bike rides, the ride duration for electric bikes is shorter than that for classic bikes.
3. While members exhibit a fairly even distribution between classic and electric bike usage, casual riders demonstrate a stronger inclination toward classic bikes.(for longer trips)
4. On the weekend rides are fairly longer, where casual riders take longer rides than their counterparts.
5. Overall the ride count for members is higher than that for the casuals but during the weekends, casuals and members take fairly equal rides altogether.
6. There is a larger number of rides in the summer months (June-August) than in other seasons.
7. For members, throughout the season, ride durations are fairly similar but for casuals, ride duration peaks during the summer months.
8. Members' ride durations are distributed across the day, but for the casuals, rides are longer during the afternoon.
9. There is a far bigger drop across August and December for casuals(x12), than that for members(x4).
10. Across every single month, members take more rides than casual riders.


## Interpretations respective to the given point observations:
1. Casual riders take significantly longer rides than members, suggesting they use bikes selectively for longer trips rather than short, frequent trips.
2. Electric bikes are associated with shorter ride durations across both rider types, suggesting they are preferred for quick trips.
3. Members show similar ride durations regardless of bike type, while casuals show significantly longer durations on classic bikes, suggesting casuals make more deliberate bike type choices based on trip length
4. Overall riders prefer bikes for longer trips during weekend, with the casuals showing stronger preference.
5. On weekends there is no work commute infering casuals prefer to take bikes for leisure than for their work, if opportunity is there.
6. Riders prefer to take bikes during the summer months than to the winter months, where casuals are more season dependent.
7. Casual ridership and ride duration drops sharply outside summer months.
8. Apart from night members, take almost equal amounts of rides throughout the day, meaning they take rides from, to, and during work and for leisure as well. There is a higher count of rides in the afternoon for the casuals, meaning, if they take bikes during weekdays, they prefer rides for for work commute or lunch breaks or so.
9. Casuals don't prefer bikes during the winter seasons. For members though, there is a fall in the ride counts during the other seasons than the summers but that is fairly smaller so they are more resilient to season change.
10. Members ride more frequently and consistently across all time periods and seasons, suggesting membership encourages habitual usage.



In [100]:
rd.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration,day_of_week,ride_minutes,month,hour,time_of_day
ride_id,,,,,,,,,,,,,,,,,,
7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member,0 days 00:13:57.477000,Tuesday,13.957950,January,17,Evening
013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member,0 days 00:05:04.344000,Saturday,5.072400,January,15,Afternoon
EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,0 days 00:11:35.500000,Thursday,11.591667,January,15,Afternoon
EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:03:34.233000,Thursday,3.570550,January,8,Morning
7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member,0 days 00:02:34.429000,Thursday,2.573817,January,8,Morning


In [105]:
rd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5399849 entries, 7569BC890583FCD7 to D2CF23B6D79232B6
Data columns (total 18 columns):
 #   Column              Dtype          
---  ------              -----          
 0   rideable_type       object         
 1   started_at          datetime64[ns] 
 2   ended_at            datetime64[ns] 
 3   start_station_name  object         
 4   start_station_id    object         
 5   end_station_name    object         
 6   end_station_id      object         
 7   start_lat           float64        
 8   start_lng           float64        
 9   end_lat             float64        
 10  end_lng             float64        
 11  member_casual       object         
 12  ride_duration       timedelta64[ns]
 13  day_of_week         object         
 14  ride_minutes        float64        
 15  month               category       
 16  hour                int32          
 17  time_of_day         category       
dtypes: category(2), datetime64[ns](2), float64(

In [109]:
# exporting the whole file to a csv format
rd.to_csv('full_data.csv', index=False)

In [42]:
# normalizing some of the data for better visualization
#per month
monthly_counts = rd.groupby(['member_casual', 'month']).size().reset_index(name='ride_count')
group_totals = monthly_counts.groupby('member_casual')['ride_count'].transform('sum')
monthly_counts['percentage'] = (monthly_counts['ride_count'] / group_totals) * 100
monthly_counts.head()
monthly_counts.to_csv('normalizedMonth.csv', index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_19812\966549131.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  monthly_counts = rd.groupby(['member_casual', 'month']).size().reset_index(name='ride_count')


In [44]:
#per week
weekly_counts = rd.groupby(['member_casual', 'day_of_week']).size().reset_index(name='ride_count')
group_totals = weekly_counts.groupby('member_casual')['ride_count'].transform('sum')
weekly_counts['percentage'] = (weekly_counts['ride_count'] / group_totals) * 100
weekly_counts.head()
weekly_counts.to_csv('normalizedweek.csv', index=False)

In [46]:
#per part of day
day_change = rd.groupby(['member_casual', 'time_of_day']).size().reset_index(name='ride_count')
group_totals = day_change.groupby('member_casual')['ride_count'].transform('sum')
day_change['percentage'] = (day_change['ride_count'] / group_totals) * 100
day_change.head()
day_change.to_csv('normalizeday.csv', index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_19812\1996816208.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  day_change = rd.groupby(['member_casual', 'time_of_day']).size().reset_index(name='ride_count')
